In [1]:
import netCDF4 as nc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import os
import xarray as xr
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import pyproj
from scipy.interpolate import griddata

In [2]:
home_path = os.path.expanduser("~")

path = '/DataFiles'
bm = xr.open_dataset(home_path + path + "/BedMachineAntarctica-v3.nc")

In [3]:
print(bm)

<xarray.Dataset> Size: 4GB
Dimensions:    (x: 13333, y: 13333)
Coordinates:
  * x          (x) int32 53kB -3333000 -3332500 -3332000 ... 3332500 3333000
  * y          (y) int32 53kB 3333000 3332500 3332000 ... -3332500 -3333000
Data variables:
    mapping    |S1 1B ...
    mask       (y, x) int8 178MB ...
    firn       (y, x) float32 711MB ...
    surface    (y, x) float32 711MB ...
    thickness  (y, x) float32 711MB ...
    bed        (y, x) float32 711MB ...
    errbed     (y, x) float32 711MB ...
    source     (y, x) int8 178MB ...
    dataid     (y, x) int8 178MB ...
    geoid      (y, x) int16 356MB ...
Attributes: (12/17)
    Conventions:                 CF-1.7
    Title:                       BedMachine Antarctica
    Author:                      Mathieu Morlighem
    version:                     03-Jun-2022 (v3.4)
    nx:                          13333.0
    ny:                          13333.0
    ...                          ...
    ymax:                        3333000
  

In [4]:
xArr = bm['x'].values
yArr = bm['y'].values
X, Y = np.meshgrid(xArr, yArr)
surface = bm['surface'].values  # 2D (y, x)
bed = bm['bed'].values
thick = bm['thickness'].values

# Define the same projection used for the grid
proj = pyproj.CRS("EPSG:3031")  # Replace with your projection
transformer = pyproj.Transformer.from_crs("EPSG:4326", proj, always_xy=True)


In [8]:
# Convert target point
target_lon, target_lat = -112.09, -79.46
#target_lon, target_lat = -90, -90
target_x, target_y = transformer.transform(target_lon, target_lat)

# Find nearest indices
ix = np.argmin(np.abs(xArr - target_x))
iy = np.argmin(np.abs(yArr - target_y))

# Lookup the surface value
value = surface[iy, ix]
print(f"Surface value at lat={target_lat}, lon={target_lon}: {value}")
print(f"Bed value at lat={target_lat}, lon={target_lon}: {bed[iy, ix]}")
print(f"Thickness value at lat={target_lat}, lon={target_lon}: {thick[iy, ix]}")


Surface value at lat=-79.46, lon=-112.09: 1780.017333984375
Bed value at lat=-79.46, lon=-112.09: -1684.769287109375
Thickness value at lat=-79.46, lon=-112.09: 3464.78662109375


In [6]:
vel = xr.open_dataset(home_path + path + "/BedMachineAntarctica-v3.nc")

In [7]:
newData = nc.Dataset(home_path + path + "/data.nc")